# core

> `PtySession` and `PtyRegistry`: asyncio pty sessions over one offset-tracked output ring


In [ ]:
#| default_exp core

Every design here serves two consumer shapes over one pty. *Streams* (a websocket handler, a renderer) want every byte in order, pushed as it arrives, with recent scrollback replayed on attach. *Cursors* (an LLM tool that reads "what's new since I last looked") want bounded-memory buffering with paging and honest drop accounting. Both are views over a single structure: the `Ring`, a byte buffer with absolute offsets. Each pty read appends to the ring; a session-wide change event wakes waiters; `read_from(offset)` serves both a stream's next chunk and a cursor's next page, and reports how many bytes fell off the back. There is no server here, no framework, and no persistence beyond the process: web exposure belongs to the embedding app (e.g. jupygate), durable sessions belong to tmux.

In [ ]:
#| export
import asyncio, itertools, logging, os, signal, time
from collections import deque
from pathlib import Path
from tempfile import mkdtemp
from fastcore.basics import patch
from ptyprocess import PtyProcess

log = logging.getLogger('ptymini')

In [ ]:
import sys
from fastcore.test import test_eq

## The ring

A `Ring` is a bounded byte buffer that never forgets *where* it is in the absolute output stream: `start` and `end` are offsets from byte 0 of everything the pty ever produced, and appends trim the oldest bytes once `max_bytes` is exceeded. `read_from(offset)` returns the bytes from that absolute position (capped by `max_bytes_out`), the new offset, and how many bytes the caller *missed* because they fell off the back — the number a cursor consumer reports as `dropped_bytes` and a stream consumer surfaces as a gap. The chunk/trim/collect logic is lifted from bgterm, whose production buffer this is.

In [ ]:
#| export
class Ring:
    "Bounded byte buffer with absolute offsets: `start`..`end` of the retained window, `end` = total bytes ever appended."
    def __init__(self, max_bytes:int=1_000_000):
        if max_bytes <= 0: raise ValueError('max_bytes must be > 0')
        self.max_bytes, self.chunks, self.start, self.end = max_bytes, deque(), 0, 0

    def __len__(self): return self.end - self.start

    def append(self, data:bytes):
        self.chunks.append((self.end, data))
        self.end += len(data)
        while len(self) > self.max_bytes:
            off, d = self.chunks[0]
            overflow = len(self) - self.max_bytes
            if overflow >= len(d):
                self.chunks.popleft()
                self.start = off + len(d)
            else:
                self.chunks[0] = (off + overflow, d[overflow:])
                self.start = off + overflow

    def read_from(self,
        offset:int,                  # Absolute position to read from; behind `start` counts as dropped
        max_bytes_out:int=None,      # Cap on returned bytes (None: everything retained)
    ):                               # (data, new_offset, dropped)
        dropped = max(0, self.start - offset)
        cur = max(offset, self.start)
        out = bytearray()
        for off, d in self.chunks:
            if off + len(d) <= cur: continue
            piece = d[max(0, cur - off):]
            if max_bytes_out is not None: piece = piece[:max_bytes_out - len(out)]
            if not piece: break
            out.extend(piece)
            if max_bytes_out is not None and len(out) >= max_bytes_out: break
        return bytes(out), cur + len(out), dropped

The offsets are the contract, so pin them: reads page deterministically, a reader behind the window learns exactly what it lost, and reading past the end returns empty without moving anything.

In [ ]:
r = Ring(max_bytes=8)
r.append(b'abcdef')
test_eq(r.read_from(0), (b'abcdef', 6, 0))
r.append(b'ghij')                      # 10 bytes total; the oldest 2 trim away
test_eq((r.start, r.end, len(r)), (2, 10, 8))
data, off, dropped = r.read_from(0)    # a reader at 0 lost the first 2 bytes
test_eq((data, off, dropped), (b'cdefghij', 10, 2))
page, off1, _ = r.read_from(2, max_bytes_out=3)   # paging: 3 bytes, then the rest
test_eq((page, off1), (b'cde', 5))
test_eq(r.read_from(off1)[0], b'fghij')
test_eq(r.read_from(off)[0], b'')      # caught up: empty, offset unmoved
r.read_from(off)

## PtySession

One spawned pty. Reads land in the ring; a change *event chain* wakes whoever is waiting (each notification replaces the event, so a waiter grabs the current one and awaits it — safe to fire from the synchronous `add_reader` callback, where an `asyncio.Condition` couldn't be). EOF — however the child dies — reaps the exit status, closes the fd, and fires the chain one last time so every waiter observes the end. Requires a running event loop; `bg` (the sync layer) supplies its own.

In [ ]:
#| export
class PtySession:
    "One pty process: reads land in a `Ring`, a change event wakes waiters, EOF reaps the exit status."
    def __init__(self,
        argv:list[str],          # Command to spawn on the pty
        cwd:str=None,            # Working directory for the child
        env:dict=None,           # Child environment; None inherits this process's
        rows:int=24, cols:int=80,
        buffer_bytes:int=1_000_000,  # Ring bound: how much recent output is retained
        name:str=None,           # Optional handle, set by `PtyRegistry`
    ):
        self.name, self.alive, self.exit_code = name, True, None
        self.ring = Ring(buffer_bytes)
        self.last_activity = time.time()
        self._evt, self._eof_evt = asyncio.Event(), asyncio.Event()
        self.proc = PtyProcess.spawn(list(argv), cwd=cwd, env=env, dimensions=(rows, cols))
        asyncio.get_running_loop().add_reader(self.proc.fd, self._on_read)

    def _notify(self):
        evt, self._evt = self._evt, asyncio.Event()
        evt.set()

    def _on_read(self):
        try: data = self.proc.read(65536)
        except (EOFError, OSError): data = b''
        if not data: return self._eof()
        self.last_activity = time.time()
        self.ring.append(data)
        self._notify()

    def _eof(self):
        self.alive = False
        asyncio.get_running_loop().remove_reader(self.proc.fd)
        self.proc.isalive()   # reaps; fills exitstatus/signalstatus
        self.exit_code = -self.proc.signalstatus if self.proc.signalstatus else self.proc.exitstatus
        try: self.proc.close()
        except OSError: pass
        self._eof_evt.set()
        self._notify()

    async def wait_change(self,
        seen_end:int=None,   # Wake when `ring.end` passes this (None: the value at call time)
        timeout:float=None,  # Max seconds to wait; None waits indefinitely
    )->bool:                 # True when new output or EOF arrived, False on timeout
        if seen_end is None: seen_end = self.ring.end
        deadline = None if timeout is None else time.monotonic() + timeout
        while self.alive and self.ring.end == seen_end:
            t = None if deadline is None else max(0, deadline - time.monotonic())
            try: await asyncio.wait_for(self._evt.wait(), t)
            except asyncio.TimeoutError: return False
        return True

    def read_from(self, offset:int, max_bytes_out:int=None): return self.ring.read_from(offset, max_bytes_out)

    def write(self, data:bytes):
        self.last_activity = time.time()
        self.proc.write(data)

    def resize(self, rows:int, cols:int): self.proc.setwinsize(rows, cols)
    def kill(self, sig:int=signal.SIGTERM): self.proc.kill(sig)

    async def wait(self)->int:
        "The exit code, once the pty dies (negative: terminating signal number)."
        await self._eof_evt.wait()
        return self.exit_code

    def model(self)->dict: return dict(name=self.name, alive=self.alive, last_activity=self.last_activity)

    async def __aenter__(self): return self
    async def __aexit__(self, *exc): await self.terminate(force=True)

Termination escalates through the polite signals before force (the ladder follows terminado's `PtyWithClients.terminate`, BSD-licensed, Copyright (c) Jupyter Development Team): a shell holding a foreground job deserves the chance to clean up. `__aexit__` takes the force path — a `with` block ending means the scope is over.


In [ ]:
#| export
@patch
async def terminate(self:PtySession, force:bool=False)->bool:
    "SIGHUP/SIGCONT/SIGINT/SIGTERM in turn, then SIGKILL if `force`; True if the child ended."
    for sig in (signal.SIGHUP, signal.SIGCONT, signal.SIGINT, signal.SIGTERM):
        if not self.proc.isalive(): return True
        try: self.kill(sig)
        except ProcessLookupError: return True
        await asyncio.sleep(self.proc.delayafterterminate)
    if not self.proc.isalive(): return True
    if force:
        self.proc.kill(signal.SIGKILL)
        await asyncio.sleep(self.proc.delayafterterminate)
    return not self.proc.isalive()

A deterministic child for the examples: `bash --norc --noprofile` with a fixed prompt, so no user rc noise reaches the assertions. `read_until` drains a client queue until a pattern shows up — containment with a timeout, never exact frames, because pty output chunking is timing-dependent by nature.

In [ ]:
BASH = ['bash', '--norc', '--noprofile', '-i']
BENV = dict(os.environ, PS1='$ ', TERM='dumb')

async def read_until(t, pat:bytes, offset:int=0, timeout=10.0):
    "Read from `offset` until `pat` appears in the accumulation (or EOF); returns (bytes, offset)."
    buf = b''
    end = time.monotonic() + timeout
    while pat not in buf:
        data, offset, _ = t.read_from(offset)
        buf += data
        if pat in buf or not (t.alive or data): break
        if not data: await t.wait_change(seen_end=offset, timeout=end - time.monotonic())
    return buf, offset


Spawn a shell, type a command, read the echo and the output back with the primitive pair — `read_from` plus `wait_change` — which is all a cursor consumer ever needs. Bytes in, bytes out; nothing decodes anywhere.


In [ ]:
t = PtySession(BASH, env=BENV)
t.write(b'echo hi $((6*7))\n')
out, off = await read_until(t, b'hi 42')
assert t.alive
out[-24:]


## Streams

`attach` is the *stream* view: an async iterator yielding bytes in order — everything retained (replay), then live output as it arrives — ending at EOF. Each attachment is just a cursor the generator owns, so any number of clients attach independently and closing the iterator (a `break`, a cancelled task) leaks nothing. A consumer that falls behind the ring resumes at the oldest retained byte; with `gaps=True` the miss is announced in-stream as a `Gap` first, so a terminal client can clear-and-redraw instead of rendering a stream with an invisible hole.

In [ ]:
#| export
class Gap(int):
    "Bytes lost to ring trimming between two yields of an `attach` stream."

@patch
async def attach(self:PtySession,
    gaps:bool=False,       # Also yield `Gap(n)` markers when the stream fell behind the ring?
    from_start:bool=True,  # Begin at the oldest retained byte (replay); False starts at the live edge
):
    "Async byte stream over the session's output: replay, then live; ends at EOF."
    off = self.ring.start if from_start else self.ring.end
    while True:
        data, off, dropped = self.read_from(off)
        if dropped and gaps: yield Gap(dropped)
        if data: yield data
        elif not self.alive: return
        else: await self.wait_change(seen_end=off)

An attachment that starts *now* still sees what happened before it existed — the replay is just the retained ring. This is what a page refresh or dropped connection costs a reattaching client: nothing.


In [ ]:
chunks = []
async for chunk in t.attach():
    chunks.append(chunk)
    if b'hi 42' in b''.join(chunks): break
assert b'echo hi' in b''.join(chunks)  # the command echo and its output both replayed from the ring
len(chunks)


Streams and cursors coexist on one session: while the attach loop above watched the live edge, the `read_until` cursor below sees the same bytes independently. Resize is an ioctl on the pty, and the shell reports its own size back — the verification is in-band, like everything on a terminal.


In [ ]:
t.resize(40, 100)
t.write(b'stty size\n')
out, off = await read_until(t, b'40 100', offset=off)
b'40 100' in out


In [ ]:
assert await t.terminate(force=True)
code = await t.wait()
test_eq(t.alive, False)
final = [c async for c in t.attach()]       # a dead session's stream: full replay, then it ends
assert b'hi 42' in b''.join(final)
code                                        # negative: killed by that signal number


A command that exits on its own delivers its code through `wait` — and the session context manager guarantees the reap, so a `with` block can't leak a pty even on an exception:

In [ ]:
async with PtySession([sys.executable, '-c', 'print("bye"); raise SystemExit(3)']) as p:
    test_eq(await p.wait(), 3)
    test_eq(p.alive, False)

The gap contract, live: a stream that isn't consumed while the child floods past the ring's bound resumes with a `Gap` announcing exactly how many bytes it missed, then the retained tail. This is what lets a terminal client clear-and-redraw instead of rendering around an invisible hole.

In [ ]:
g = PtySession(['bash', '-c', 'printf a; sleep 0.3; printf "%0999d" 7'], buffer_bytes=64)
s = g.attach(gaps=True)
first = await anext(s)
test_eq(first, b'a')
await g.wait()                    # the flood happens while nobody reads the stream
nxt = await anext(s)
assert isinstance(nxt, Gap)
test_eq(int(nxt), 935)            # 1 byte read + 999 flooded - 64 retained
rest = b''.join([c async for c in s])
test_eq(len(rest), 64)
int(nxt), rest[-4:]

## The registry

`PtyRegistry` makes sessions *named*: get-or-create by name, so a name is a stable handle a client can come back to; auto-numbering when the caller doesn't care; reap-all at shutdown (also the `async with` exit). Creation takes an `argv` override, `env`/`appendenv` (replace/overlay the inherited environment), `cwd` — plus one affordance with real content: `rc`, injected shell setup. A host spawning a user's shell with prompt integration (sentinel boundaries, emptied prompts) needs that rc text to become a *file on this machine*, so the registry writes it to a private directory and substitutes `{rcfile}` and `{rcdir}` into the argv and env values. The file is named `.zshrc` so one mechanism serves both shells: bash takes the file (`--rcfile {rcfile}`), zsh takes the directory (`ZDOTDIR={rcdir}`). Privilege policy (sudo wrapping and the like) deliberately stays with the embedding app: prefix the argv you pass.


In [ ]:
#| export
DEFAULT_ARGV = [os.environ.get('SHELL') or 'bash', '-i']

class PtyRegistry:
    "Named-session registry: get-or-create, list, terminate. An embedding app or a test drives this."

    def __init__(self,
        argv:list[str]=None,      # Default spawn command; creation requests may override it
        cull_timeout:float=0,     # Seconds of inactivity before a terminal is reaped (0 disables)
    ):
        self.argv = argv or DEFAULT_ARGV
        self.cull_timeout, self.terms = cull_timeout, {}

    def _next_name(self): return next(str(i) for i in itertools.count(1) if str(i) not in self.terms)

    async def create(self,
        name:str=None,           # Existing name reattaches; None auto-numbers
        argv:list[str]=None,     # Spawn command; the registry default if None
        cwd:str=None,            # Working directory for the child
        env:dict=None,           # Replaces the inherited environment
        appendenv:dict=None,     # Overlays the base environment
        rc:str=None,             # Shell setup text, written to a private dir; `{rcfile}`/`{rcdir}` substitute into argv and env values
        rows:int=24, cols:int=80,
    )->PtySession:
        "Get session `name` if it exists, else spawn one (auto-named when `name` is None)."
        if name and name in self.terms: return self.terms[name]
        name = name or self._next_name()
        argv = list(argv or self.argv)
        if rc is not None:
            rcdir = mkdtemp(prefix='ptymini-rc-')
            rcfile = str(Path(rcdir)/'.zshrc')  # one name serves bash (--rcfile {rcfile}) and zsh (ZDOTDIR={rcdir})
            Path(rcfile).write_text(rc)
            subst = lambda s: s.replace('{rcfile}', rcfile).replace('{rcdir}', rcdir)
            argv = [subst(a) for a in argv]
            if env: env = {k: subst(v) for k, v in env.items()}
            if appendenv: appendenv = {k: subst(v) for k, v in appendenv.items()}
        full_env = dict(os.environ if env is None else env) | (appendenv or {})  # `env` replaces the inherited environment; `appendenv` overlays the base
        t = PtySession(argv, cwd=cwd, env=full_env, rows=rows, cols=cols, name=name)
        self.terms[name] = t
        return t

    async def delete(self, name:str):
        t = self.terms.pop(name)
        await t.terminate(force=True)

    def cull_ready(self)->list[str]:
        "Names of terminals past the inactivity timeout (empty when culling is disabled)."
        if not self.cull_timeout: return []
        now = time.time()
        return [n for n, t in self.terms.items() if now - t.last_activity > self.cull_timeout]

    async def cull(self):
        for n in self.cull_ready():
            log.info('culling inactive session %s', n)
            await self.delete(n)

    async def shutdown(self):
        "Reap every terminal; nothing survives the registry."
        await asyncio.gather(*[self.delete(n) for n in list(self.terms)], return_exceptions=True)

    def get(self, name:str)->PtySession|None: return self.terms.get(name)
    def values(self): return self.terms.values()
    def __len__(self): return len(self.terms)

    async def __aenter__(self): return self
    async def __aexit__(self, *exc): await self.shutdown()

Get-or-create by name: asking for a name that exists reattaches (that's what makes the name a stable handle across page refreshes); a fresh name or none spawns.

In [ ]:
terms = PtyRegistry(argv=BASH)
ta = await terms.create(env=BENV)
test_eq(ta.name, '1')
assert (await terms.create(name='1')) is ta          # existing name: same session back
tb = await terms.create(name='scratch', env=BENV)
[t.model()['name'] for t in terms.values()]


The rc affordance, round-tripped: the rc text defines a marker alias, `{rcfile}` lands in the argv, and the spawned shell has the alias — proof the injected setup ran on the gateway's side.

In [ ]:
tr = await terms.create(argv=['bash', '--noprofile', '--rcfile', '{rcfile}', '-i'],
    rc="PS1='$ '\nalias hi='echo rc-worked'", env=dict(BENV))
tr.write(b'hi\n')
out, _ = await read_until(tr, b'rc-worked')
assert b'rc-worked' in out
tr.name

Culling is a decision plus a sweep, split so the decision is testable without a clock: `cull_ready` names the terminals past the inactivity timeout, `cull` reaps them. An embedding app runs the sweep (`cull_loop`) as a task while it serves; activity means pty reads or client writes, stamped in `_on_read` and `write`.

In [ ]:
test_eq(terms.cull_ready(), [])          # disabled by default
terms.cull_timeout = 3600
ta.last_activity -= 7200                 # stub the clock: ta has been idle two hours
test_eq(terms.cull_ready(), ['1'])
await terms.cull()
assert terms.get('1') is None and terms.get('scratch') is not None
len(terms)

In [ ]:
#| export
async def cull_loop(reg:PtyRegistry, interval:float=300):
    "Periodic cull sweep; an embedding app runs this as a task while it serves."
    while True:
        await asyncio.sleep(interval)
        await reg.cull()

In [ ]:
await terms.shutdown()
test_eq(len(terms), 0)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()